In [1]:
from dotenv import load_dotenv
import os, sys, json, re, subprocess, tempfile

load_dotenv()

from astrapy import DataAPIClient
from github import Auth, Github
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.agents import create_agent

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
client = DataAPIClient(os.getenv("ASTRA_DB_APPLICATION_TOKEN"))
db = client.get_database(os.getenv("ASTRA_DB_API_ENDPOINT"))
collection = db.get_collection("codeguardian_style_corpus")

def retrieve_style_context(diff_text: str, top_k: int = 5) -> str:
    query_vector = embeddings.embed_query(diff_text)
    results = collection.find(sort={"$vector": query_vector}, limit=top_k)
    return "\n\n".join(f"[{r['type']} — {r['source']}]\n{r['text']}" for r in results)

gh = Github(auth=Auth.Token(os.getenv("GITHUB_PAT")))
repo = gh.get_repo("PrashantAghara/fastapi")
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

print("Setup complete")

d:\resume-projects\codegaurdian\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18190.80it/s]


Setup complete


In [2]:
def get_changed_line_ranges(patch: str) -> list[tuple[int, int]]:
    ranges = []
    for match in re.finditer(r"@@ -\d+,?\d* \+(\d+),?(\d*) @@", patch):
        start = int(match.group(1))
        length = int(match.group(2)) if match.group(2) else 1
        ranges.append((start, start + length - 1))
    return ranges

def get_pr_diff_text(pr, filenames: list[str]) -> str:
    diff_parts = []
    for f in pr.get_files():
        if f.filename in filenames:
            diff_parts.append(f"--- {f.filename} ---\n{f.patch}")
    return "\n\n".join(diff_parts)

def get_full_file_content(pr, filename: str) -> str:
    return repo.get_contents(filename, ref=pr.head.sha).decoded_content.decode("utf-8")

In [3]:
current_pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == "test/lint-violation")
py_filenames = [f.filename for f in current_pr.get_files() if f.filename.endswith(".py")]

In [4]:
def run_ruff_on_file(filename: str, full_content: str) -> list[dict]:
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w", encoding="utf-8") as tmp:
        tmp.write(full_content)
        tmp_path = tmp.name
    result = subprocess.run(["ruff", "check", tmp_path, "--output-format=json"], capture_output=True, text=True)
    os.unlink(tmp_path)
    try:
        return json.loads(result.stdout) if result.stdout else []
    except json.JSONDecodeError:
        return []

def filter_to_diff(findings: list[dict], ranges: list[tuple[int, int]], row_key=lambda f: f["location"]["row"]) -> list[dict]:
    return [f for f in findings if any(s <= row_key(f) <= e for s, e in ranges)]

@tool
def static_analysis_tool(filename: str) -> dict:
    """Run ruff against this PR's version of a file, scoped to only the changed lines."""
    file_obj = next(f for f in current_pr.get_files() if f.filename == filename)
    full_content = get_full_file_content(current_pr, filename)
    raw = run_ruff_on_file(filename, full_content)
    ranges = get_changed_line_ranges(file_obj.patch)
    return {"filename": filename, "findings": filter_to_diff(raw, ranges)}

static_analysis_agent = create_agent(
    model=llm,
    tools=[static_analysis_tool],
    system_prompt=(
        "You are a Static Analysis Agent reviewing a pull request. "
        "Call static_analysis_tool once per given filename. "
        "When summarizing, use the EXACT 'location.row' value as the Line number — never estimate. "
        "Summarize findings grouped by severity, end with a one-line verdict: PASS, PASS_WITH_WARNINGS, or FAIL."
    ),
)

In [5]:
BASELINE_RULES = """
- All function signatures must have type hints on parameters and return values
- Public functions must have a docstring describing purpose, args, and return value
- Function and variable names must be descriptive, not abbreviated (e.g. `user_id` not `uid`)
- Avoid nesting conditionals more than 3 levels deep — prefer early returns
"""

baseline_prompt = """You are reviewing a pull request diff against ONLY these baseline rules:
{baseline_rules}

Diff:
{diff}

For each violation: filename, line (from diff hunk header), comment, severity (info/warning).
If none, say so explicitly."""

project_context_prompt = """You are reviewing a pull request diff against ONLY the conventions evident in this project's actual codebase (retrieved below) — not general best practices.

Retrieved project context:
{retrieved_context}

Diff:
{diff}

Identify anything in the diff that deviates from patterns clearly shown in the retrieved context above.
For each: filename, line (from diff hunk header), comment, severity (info/warning).
If the context doesn't clearly support a finding, say so rather than guessing."""

def run_style_agent_two_pass(pr, filenames: list[str]) -> str:
    diff_text = get_pr_diff_text(pr, filenames)
    retrieved_context = retrieve_style_context(diff_text)

    baseline_result = llm.invoke(baseline_prompt.format(baseline_rules=BASELINE_RULES, diff=diff_text))
    context_result = llm.invoke(project_context_prompt.format(retrieved_context=retrieved_context, diff=diff_text))

    merge_prompt = f"""Combine these two independent review passes into one final report. Keep each finding's source labeled.

BASELINE PASS RESULTS:
{baseline_result.content}

PROJECT-CONTEXT PASS RESULTS:
{context_result.content}

Output a combined list of findings (deduplicated if any overlap), each labeled [baseline] or [project-context], then end with one overall verdict: PASS, PASS_WITH_WARNINGS, or FAIL."""

    return llm.invoke(merge_prompt).content

In [6]:
def run_bandit_on_file(filename: str, full_content: str) -> list[dict]:
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w", encoding="utf-8") as tmp:
        tmp.write(full_content)
        tmp_path = tmp.name
    result = subprocess.run(["bandit", "-f", "json", tmp_path], capture_output=True, text=True)
    os.unlink(tmp_path)
    try:
        data = json.loads(result.stdout)
        return data.get("results", [])
    except json.JSONDecodeError:
        return []

def filter_bandit_to_diff(findings: list[dict], ranges: list[tuple[int, int]]) -> list[dict]:
    # bandit uses "line_number", not the nested "location.row" shape ruff uses
    return [f for f in findings if any(s <= f["line_number"] <= e for s, e in ranges)]

In [7]:
@tool
def security_analysis_tool(filename: str) -> dict:
    """Run bandit against this PR's version of a file, scoped to only the changed lines, to find security issues."""
    file_obj = next(f for f in current_pr.get_files() if f.filename == filename)
    full_content = get_full_file_content(current_pr, filename)
    raw = run_bandit_on_file(filename, full_content)
    ranges = get_changed_line_ranges(file_obj.patch)
    filtered = filter_bandit_to_diff(raw, ranges)
    return {
        "filename": filename,
        "findings": [
            {
                "line": f["line_number"],
                "issue": f["issue_text"],
                "severity": f["issue_severity"],
                "confidence": f["issue_confidence"],
                "test_id": f["test_id"],
            }
            for f in filtered
        ],
    }

security_agent = create_agent(
    model=llm,
    tools=[security_analysis_tool],
    system_prompt=(
        "You are a Security Agent reviewing a pull request for security risks. "
        "Call security_analysis_tool once per given filename. "
        "Use the EXACT 'line' value from each finding — never estimate. "
        "Summarize findings grouped by severity, and end with a one-line verdict: PASS, PASS_WITH_WARNINGS, or FAIL. "
        "Any 'high' severity finding should always result in FAIL, regardless of how few findings there are."
    ),
)

In [8]:
def run_all_agents(pr) -> dict:
    global current_pr
    current_pr = pr

    py_filenames = [f.filename for f in pr.get_files() if f.filename.endswith(".py")]

    static_result = static_analysis_agent.invoke({
        "messages": [{"role": "user", "content": f"Changed Python files: {py_filenames}"}]
    })["messages"][-1].content

    style_result = run_style_agent_two_pass(pr, py_filenames)

    security_result = security_agent.invoke({
        "messages": [{"role": "user", "content": f"Changed Python files: {py_filenames}"}]
    })["messages"][-1].content

    return {
        "static_analysis": static_result,
        "style": style_result,
        "security": security_result,
    }

In [9]:
summarizer_prompt = """You are a Summarizer Agent producing a final PR review summary from three independent agent reports below.

STATIC ANALYSIS REPORT:
{static_analysis}

STYLE REPORT:
{style}

SECURITY REPORT:
{security}

Produce:
1. A concise PR summary (2-3 sentences) describing what changed
2. A suggested commit message (conventional-commits style, e.g. "fix: ...", "feat: ...")
3. A consolidated list of all findings across the three reports, grouped by severity
4. An OVERALL VERDICT: APPROVE, REQUEST_CHANGES, or ESCALATE
   - ESCALATE only if the security report has any high-severity finding
   - REQUEST_CHANGES if there are any warnings/errors from static analysis or style
   - APPROVE only if all three reports are clean
"""

def run_summarizer(agent_results: dict) -> str:
    result = llm.invoke(summarizer_prompt.format(**agent_results))
    return result.content

In [10]:
security_pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == "test/security-violation")

results = run_all_agents(security_pr)
final_summary = run_summarizer(results)

print(final_summary)

**PR Summary**  
This change adds a new public function `run_dynamic_query` to `fastapi/applications.py`. The implementation introduces a hard‑coded secret and uses `eval` on user‑provided input, while also lacking type hints and a docstring.

**Suggested Commit Message**  
```
feat: add run_dynamic_query endpoint

- Introduces dynamic query execution (currently uses eval)
- Includes placeholder secret (hard‑coded)
- Missing type hints and docstring (to be added)
```

**Consolidated Findings**

| Severity | File | Line(s) | Finding |
|----------|------|---------|---------|
| **Warning** | `fastapi/applications.py` | 4775 | `run_dynamic_query` has no type hints on its parameter or return value. |
| **Warning** | `fastapi/applications.py` | 4775 | `run_dynamic_query` lacks a docstring describing its purpose, arguments, and return value. |
| **Warning** | `fastapi/applications.py` | 4775 (first added line) | Hard‑coded secret `SECRET_KEY = "sk_live_hardcoded_1234567890abcdef"` – secrets s

In [11]:
for branch in ["test/lint-violation", "test/style-violation"]:
    pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == branch)
    print(f"\n{'='*20} {branch} {'='*20}")
    results = run_all_agents(pr)
    print(run_summarizer(results))


==================== test/lint-violation ====================
**PR Summary**  
The change introduces an `import uuid` statement in `fastapi/applications.py` after the function definitions. The import is never used, violating both static‑analysis rules and the project’s import‑ordering conventions.

**Suggested Commit Message**  
```
fix: remove stray unused uuid import from fastapi/applications.py
```

**Consolidated Findings**

| Severity | Source | File | Line | Description |
|----------|--------|------|------|-------------|
| **Error** | Static Analysis | fastapi/applications.py | 4775 | Unused import `uuid` (`F401`). The symbol is never referenced. |
| **Warning** | Style Review | fastapi/applications.py | 4775 | Import added after function definitions and not used; breaks the project’s convention of placing all imports at the top of the module. |

*Security Report*: No findings.

**OVERALL VERDICT:** **REQUEST_CHANGES**  
(Static‑analysis error and style warning require correctio